In [22]:
import os
import argparse
import pandas as pd
import numpy as np
from tqdm import tqdm
import itertools
import functools
from multiprocessing import Pool
import time
from PIL import Image
import psutil
import math
import matplotlib.pyplot as plt

In [24]:
# Data Options
path_to_images = "/home/nthom/Documents/datasets/CelebA/Img/img_align_celeba/"
# identity_labels_path = "/home/nthom/Documents/datasets/CelebA/Anno/identity_CelebA.csv"
# attribute_labels_path = "/home/nthom/Documents/datasets/CelebA/Anno/list_attr_celeba.txt"
identity_labels_path = "./preprocessed_data/pruned_by_num_samples/identity_CelebA_min-30.csv"
attribute_labels_path = "./preprocessed_data/pruned_by_num_samples/list_attr_celeba_min-30.csv"

# Preprocessing Options
preproc_save_path = "./preprocessed_data/"

In [25]:
# Read in data frames
identity_labels_df = pd.read_csv(identity_labels_path)
# attribute_labels_df = pd.read_csv(attribute_labels_path, sep=" ", skiprows=1)
attribute_labels_df = pd.read_csv(attribute_labels_path)

In [26]:
def determine_presence_of_attribute(identity):
    # Create list that will ultimately be output
    current_identity_list = [identity]
    # Get all rows in identity/image name dataframe for which the "identity_id" column is equal to the input identity parameter.
    # The convert all of these values into a list
    current_identity_image_names = identity_labels_df[identity_labels_df['identity_id'] == identity].image_name.values.tolist()
    # Get all rows in the image name/attribute label dataframe for which some value in the current_identity_image_names is equal to a value in the "image_name" column
    current_identity_attributes_df = attribute_labels_df[attribute_labels_df.image_name.isin(current_identity_image_names)]
    current_identity_attributes_df_length = len(current_identity_attributes_df.index)
    # Iterate over each attribute in the current_identity_attributes_df
    for column in current_identity_attributes_df.drop("image_name",axis=1):
        # Create a list that represents the current column
        current_column_df = current_identity_attributes_df[column]
        # Get the total number of samples in the column (this could be moved out of the for loop)
        # Get all of the positive values
        current_column_df = current_column_df[current_column_df.isin([1])]

        # Compute a ratio of positive to negative values and mark the attribute as existing in the identity if the ratio is over some threshold value
        ratio = (len(current_column_df.index)/ current_identity_attributes_df_length)
        if ratio >= 0.2:
            current_identity_list.append(1)
        else:
            current_identity_list.append(0)
    return current_identity_list

start_time = time.time()
with Pool(16) as p:
    identity_attribute_list = p.map(determine_presence_of_attribute, identity_labels_df.identity_id.unique())
print(f"Run Time: {(time.time() - start_time)} sec.")

identity_attribute_list = []
for id in tqdm(identity_labels_df.identity_id.unique()):
    output_list = determine_presence_of_attribute(id)
    identity_attribute_list.append(output_list)

Run Time: 2.625347375869751 sec.


In [ ]:
# def determine_similar_identities(attribute_count_dict: 0 , identity):
def determine_similar_identities(identity):
    current_identity_df = identity_attributes_df[identity_attributes_df.identity_id == identity]
    current_identity_column_values = current_identity_df.drop("identity_id", axis=1).values.tolist()

    current_identity_similar_identities_list = [identity]

    identities_df = identity_attributes_df[identity_attributes_df['identity_id'] != identity]

    # num_attributes = len(attributes_to_keep)
    num_positive_attributes = current_identity_df.drop("identity_id", 1).sum().sum()
    for other_identity in identities_df.identity_id.unique():
        num_agreeing_attributes = 0
        other_identity_df = identities_df[identities_df.identity_id == other_identity]
        other_identity_column_values = other_identity_df.drop("identity_id", axis=1).values.tolist()
        for index, column in enumerate(attributes_to_keep):
            if (current_identity_column_values[0][index] == 1 and other_identity_column_values[0][index] == 1):
                # attribute_count_dict[column] += 1
                num_agreeing_attributes += 1
        ratio = num_agreeing_attributes / num_positive_attributes
        if ratio >= .10:
            current_identity_similar_identities_list.append(other_identity_df.identity_id.values.tolist()[0])
    return current_identity_similar_identities_list
    # return attribute_count_dict

identity_attributes_df = pd.DataFrame(identity_attribute_list, columns=["identity_id", '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young'])

identity_attributes_df = identity_attributes_df.drop("Male", 1)

for column in identity_attributes_df.drop("identity_id", 1):
    if column not in attributes_to_keep:
        identity_attributes_df = identity_attributes_df.drop(column, 1)

# attribute_count_dict = {'Arched_Eyebrows': 0, 'Bald': 0, 'Bangs': 0, 'Black_Hair': 0, 'Blond_Hair': 0, 'Brown_Hair': 0, 'Chubby': 0, 'Double_Chin': 0, 'Eyeglasses': 0, 'Gray_Hair': 0, 'Male': 0, 'Mustache': 0, 'Narrow_Eyes': 0, 'Receding_Hairline': 0, 'Straight_Hair': 0, 'Wavy_Hair': 0}

# func = functools.partial(determine_similar_identities, attribute_count_dict)

# start_time = time.time()
# for id in tqdm(identity_attributes_df.identity_id.unique()[:16]):
#     func(id)
# print(f"Run Time: {time.time() - start_time}")

start_time = time.time()
with Pool(16) as p:
    # attribute_count_list = p.map(func, identity_attributes_df.identity_id.unique())
    similar_identities_list = p.map(determine_similar_identities, identity_attributes_df.identity_id.unique())
print(f"Run Time: {time.time() - start_time}")

similar_identities_df = pd.DataFrame(similar_identities_list)
similar_identities_df = similar_identities_df.T
similar_identities_df_header = similar_identities_df.iloc[0]
similar_identities_df = similar_identities_df[1:]
similar_identities_df.columns = similar_identities_df_header
similar_identities_df_rows = similar_identities_df.shape[0]
similar_identities_df_columns = similar_identities_df.shape[1]

In [ ]:
print(f"Num Pairs: {similar_identities_df.count().sum()}")

num_id_w_zero = 0
for column in similar_identities_df:
    if (len(similar_identities_df.index) - similar_identities_df[column].isnull().sum()) == 0:
        num_id_w_zero += 1
print(f'Number of IDs: {similar_identities_df.shape[1]}')
print(f"Num IDs w/ No Pairs: {num_id_w_zero}")
print(f"Ratio: {num_id_w_zero/similar_identities_df.shape[1]}")

In [ ]:
similar_identities_df.to_csv("./preprocessed_data/similar_identities/similar_identities_samp_thresh-30_attribs-40_intersim-25_comm_feat-100.csv", index=False)

In [ ]:
similar_identities_df = pd.read_csv("./preprocessed_data/similar_identities/similar_identities_samp_thresh-30_attribs-40_intersim-25_comm_feat-100.csv")

column_identity_images_list = []
value_identity_images_list = []
for column in similar_identities_df:
    current_column_df = similar_identities_df[column]
    if (current_column_df.isnull().sum()) == 821:
        continue
    column_identity_images_list = identity_labels_df[identity_labels_df['identity_id'] == int(float(column))].image_name.values.tolist()
    current_column_im = Image.open(path_to_images + column_identity_images_list[0])
    for value in current_column_df:
        value_identity_images_list = identity_labels_df[identity_labels_df['identity_id'] == int(float(value))].image_name.values.tolist()

        images_list = []

        for image in column_identity_images_list:
            images_list.append(image)
        for image in value_identity_images_list:
            images_list.append(image)
        images_count = len(images_list)
        grid_size = math.ceil(math.sqrt(images_count))
        fig, axes = plt.subplots(grid_size, grid_size, figsize=(40, 40))
        current_file_number=0
        for image_filename in images_list:
            x_position = current_file_number % grid_size
            y_position = current_file_number // grid_size
            plt_image = plt.imread(path_to_images + images_list[current_file_number])
            axes[x_position, y_position].imshow(plt_image)
            current_file_number += 1
        plt.subplots_adjust(left=0.0, right=1.0, bottom=0.0, top=1.0)
        plt.savefig(f"./preprocessed_data/similar_identity_grids/{column}-{value}.png")
        break

        # value_column_im = Image.open(path_to_images + value_identity_images_list[0])
        # current_column_im.show()
        # value_column_im.show()
        # x=1
        # for proc in psutil.process_iter():
        #     if proc.name() == "display":
        #         proc.kill()

In [117]:
# all attributes
# attributes_to_keep = ['5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young']

# male leaning select attributes
# attributes_to_keep = ['5_o_Clock_Shadow', 'Arched_Eyebrows', 'Bags_Under_Eyes', 'Bald', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Hat', 'Young']

# female leaning select attributes
attributes_to_keep = ['Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Mouth_Slightly_Open', 'Narrow_Eyes', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young']

# female leaning select attributes 2
# attributes_to_keep = ['Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Mouth_Slightly_Open', 'Narrow_Eyes', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Rosy_Cheeks', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Young']

# select attributes
# attributes_to_keep = ['Arched_Eyebrows', 'Bags_Under_Eyes', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Gray_Hair', 'High_Cheekbones', 'Mouth_Slightly_Open', 'Narrow_Eyes', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Smiling', 'Straight_Hair', 'Wavy_Hair']

identity_attributes_df = pd.DataFrame(identity_attribute_list, columns=["identity_id", '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young'])

for column in identity_attributes_df.drop("identity_id", axis=1):
    if column not in attributes_to_keep:
        identity_attributes_df = identity_attributes_df.drop(column, axis=1)

In [118]:
def determine_dissimilar_identities(identity):
    current_identity_df = identity_attributes_df[identity_attributes_df.identity_id == identity]
    current_identity_column_values = current_identity_df.drop("identity_id", axis=1).values.tolist()

    current_identity_dissimilar_identities_list = [identity]

    identities_df = identity_attributes_df[identity_attributes_df['identity_id'] != identity]

    num_positive_attributes = current_identity_df.drop("identity_id", axis=1).sum().sum()
    for other_identity in identities_df.identity_id.unique():
        num_disagreeing_attributes = 0
        other_identity_df = identities_df[identities_df.identity_id == other_identity]
        other_identity_column_values = other_identity_df.drop("identity_id", axis=1).values.tolist()
        for index, column in enumerate(attributes_to_keep):
            if ((current_identity_column_values[0][index] - other_identity_column_values[0][index]) != 0):
                num_disagreeing_attributes += 1
        ratio = num_disagreeing_attributes / num_positive_attributes
        if ratio >= .75:
            current_identity_dissimilar_identities_list.append(other_identity_df.identity_id.values.tolist()[0])
    return current_identity_dissimilar_identities_list

# start_time = time.time()
# for id in tqdm(identity_attributes_df.identity_id.unique()):
#     determine_dissimilar_identities(id)
# print(f"Run Time: {(time.time() - start_time)/60} min.")

start_time = time.time()
with Pool(16) as p:
    dissimilar_identities_list = p.map(determine_dissimilar_identities, identity_attributes_df.identity_id.unique())
print(f"Run Time: {(time.time() - start_time)/60} min.")

dissimilar_identities_df = pd.DataFrame(dissimilar_identities_list)
dissimilar_identities_df = dissimilar_identities_df.T
dissimilar_identities_df_header = dissimilar_identities_df.iloc[0]
dissimilar_identities_df = dissimilar_identities_df[1:]
dissimilar_identities_df.columns = dissimilar_identities_df_header
dissimilar_identities_df_rows = dissimilar_identities_df.shape[0]
dissimilar_identities_df_columns = dissimilar_identities_df.shape[1]

Run Time: 5.232210810979208 min.


In [119]:
attributes_to_keep = ['5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young']

identity_attributes_df = pd.DataFrame(identity_attribute_list, columns=["identity_id", '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young'])

for column in identity_attributes_df.drop("identity_id", axis=1):
    if column not in attributes_to_keep:
        identity_attributes_df = identity_attributes_df.drop(column, axis=1)

In [120]:
print(f"Num Pairs: {dissimilar_identities_df.count().sum()}")

num_id_w_zero = 0
for column in dissimilar_identities_df:
    if (len(dissimilar_identities_df.index) - dissimilar_identities_df[column].isnull().sum()) == 0:
        num_id_w_zero += 1
print(f'Number of IDs: {dissimilar_identities_df.shape[1]}')
print(f"Num IDs w/ No Pairs: {num_id_w_zero}")
print(f"Ratio: {num_id_w_zero/dissimilar_identities_df.shape[1]}")

Num Pairs: 3269536
Number of IDs: 2360
Num IDs w/ No Pairs: 0
Ratio: 0.0


In [129]:
def sort_pairs(column):
    return_list = [[], [], []]
    column_id_male_flag = identity_attributes_df[identity_attributes_df["identity_id"]==int(float(column))].Male.values[0]
    column_id = int(float(column))
    for value in dissimilar_identities_df[column].dropna():
        value_id_male_flag = identity_attributes_df[identity_attributes_df["identity_id"]==int(float(value))].Male.values[0]
        value_id = int(float(value))
        flag_sum = column_id_male_flag + value_id_male_flag

        if flag_sum==2:
            return_list[0].append(f"{column_id},{value_id}")
        elif flag_sum == 0:
            return_list[1].append(f"{column_id},{value_id}")
        else:
            return_list[2].append(f"{column_id},{value_id}")
    return return_list

start_time = time.time()
with Pool(16) as p:
    return_lists = p.map(sort_pairs, dissimilar_identities_df.columns)
print(f"Run Time: {(time.time() - start_time)/60} min.")

pairs_dict = {"Male": [], "Female": [], "Mixed": []}
for list_item in return_lists:
    for index, list_category in enumerate(list_item):
        for pair in list_category:
            if index == 0:
                pairs_dict["Male"].append(pair)
            elif index == 1:
                pairs_dict["Female"].append(pair)
            else:
                pairs_dict["Mixed"].append(pair)

male_diff_from_max = len(pairs_dict["Mixed"]) - len(pairs_dict["Male"])
female_diff_from_max = len(pairs_dict["Mixed"]) - len(pairs_dict["Female"])
for i in range(male_diff_from_max):
    pairs_dict["Male"].append(np.nan)
for i in range(female_diff_from_max):
    pairs_dict["Female"].append(np.nan)

sorted_dissimilar_identities_df = pd.DataFrame()
sorted_dissimilar_identities_df["Male"] = pairs_dict["Male"]
sorted_dissimilar_identities_df["Female"] = pairs_dict["Female"]
sorted_dissimilar_identities_df["Mixed"] = pairs_dict["Mixed"]

sorted_dissimilar_identities_df.to_csv("./preprocessed_data/sorted_dissimilar_identities/sorted_dissimilar_identities_samp_thresh-30_attribs-32_presence_20_comm_feat-85.csv", index=False)

Run Time: 1.2960016926129658 min.


In [ ]:
dissimilar_identities_df.to_csv("./preprocessed_data/dissimilar_identities/dissimilar_identities_samp_thresh-30_attribs-39_intersim-10_comm_feat-90.csv", index=False)

In [ ]:
identity_attributes_df = pd.DataFrame(identity_attribute_list, columns=["identity_id", '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young'])

dissimilar_identities_df = pd.read_csv("./preprocessed_data/dissimilar_identities/dissimilar_identities_samp_thresh-30_attribs-39_intersim-10_comm_feat-90.csv")

In [63]:
total = len(identity_attributes_df.index)
male_count = identity_attributes_df["Male"].sum()
female_count = total-male_count

print(f"Total Identities: {total}")
print(f"Male Identities: {male_count}")
print(f"Female Identities: {female_count}")
print(f"Ratio of Male to Female: {male_count/female_count}")

Total Identities: 2360
Male Identities: 787
Female Identities: 1573
Ratio of Male to Female: 0.5003178639542276


In [121]:
male_male_count = 0
female_female_count = 0
mixed_count = 0

# for column in tqdm(dissimilar_identities_df):
#     column_id_male_flag = identity_attributes_df[identity_attributes_df["identity_id"]==int(float(column))].Male.values[0]
#     for value in dissimilar_identities_df[column].dropna():
#         value_id_male_flag = identity_attributes_df[identity_attributes_df["identity_id"]==int(float(value))].Male.values[0]
#         flag_sum = column_id_male_flag + value_id_male_flag
#
#         if flag_sum==2:
#             male_male_count += 1
#         elif flag_sum == 0:
#             female_female_count += 1
#         else:
#             mixed_count += 1

def count_gendered_pairs(column):
    return_list = [0, 0, 0]
    column_id_male_flag = identity_attributes_df[identity_attributes_df["identity_id"]==int(float(column))].Male.values[0]
    for value in dissimilar_identities_df[column].dropna():
        value_id_male_flag = identity_attributes_df[identity_attributes_df["identity_id"]==int(float(value))].Male.values[0]
        flag_sum = column_id_male_flag + value_id_male_flag

        if flag_sum==2:
            return_list[0] += 1
        elif flag_sum == 0:
            return_list[1] += 1
        else:
            return_list[2] += 1
    return return_list

start_time = time.time()
with Pool(16) as p:
    return_lists = p.map(count_gendered_pairs, dissimilar_identities_df.columns)
print(f"Run Time: {(time.time() - start_time)/60} min.")

for value in return_lists:
    male_male_count += value[0]
    female_female_count += value[1]
    mixed_count += value[2]

Run Time: 1.330943767229716 min.


In [104]:
print("*** Select Male Leaning Attributes 85% Dissimilar (3 attributes different) ***")
print(f"Total Pairs: {mixed_count + male_male_count + female_female_count}")
print(f"Male Pairs: {male_male_count}")
print(f"Female Pairs: {female_female_count}")
print(f"Mixed Pairs: {mixed_count}")
print(f"Ratio of Male to Female Pairs: {male_male_count/female_female_count}")
print(f"Ratio of Female to Male Pairs: {female_female_count/male_male_count}")
print(f"Ratio of Gendered to Mixed pairs: {(male_male_count + female_female_count) / mixed_count}")

*** Select Male Leaning Attributes 85% Dissimilar (3 attributes different) ***
Total Pairs: 2759633
Male Pairs: 276547
Female Pairs: 529089
Mixed Pairs: 1953997
Ratio of Male to Female Pairs: 0.5226852193109288
Ratio of Female to Male Pairs: 1.9131973950178451
Ratio of Gendered to Mixed pairs: 0.41230155419890613


In [110]:
print("*** Select Female Leaning Attributes 2 85% Dissimilar (3 attributes different) ***")
print(f"Total Pairs: {mixed_count + male_male_count + female_female_count}")
print(f"Male Pairs: {male_male_count}")
print(f"Female Pairs: {female_female_count}")
print(f"Mixed Pairs: {mixed_count}")
print(f"Ratio of Male to Female Pairs: {male_male_count/female_female_count}")
print(f"Ratio of Female to Male Pairs: {female_female_count/male_male_count}")
print(f"Ratio of Gendered to Mixed pairs: {(male_male_count + female_female_count) / mixed_count}")

*** Select Female Leaning Attributes 2 85% Dissimilar (3 attributes different) ***
Total Pairs: 2580855
Male Pairs: 352027
Female Pairs: 298745
Mixed Pairs: 1930083
Ratio of Male to Female Pairs: 1.178352775778674
Ratio of Female to Male Pairs: 0.8486422916424025
Ratio of Gendered to Mixed pairs: 0.33717306457805185


In [122]:
print("*** Select Female Leaning Attributes 75% Dissimilar (5 attributes different) ***")
print(f"Total Pairs: {mixed_count + male_male_count + female_female_count}")
print(f"Male Pairs: {male_male_count}")
print(f"Female Pairs: {female_female_count}")
print(f"Mixed Pairs: {mixed_count}")
print(f"Ratio of Male to Female Pairs: {male_male_count/female_female_count}")
print(f"Ratio of Female to Male Pairs: {female_female_count/male_male_count}")
print(f"Ratio of Gendered to Mixed pairs: {(male_male_count + female_female_count) / mixed_count}")

*** Select Female Leaning Attributes 75% Dissimilar (5 attributes different) ***
Total Pairs: 3269536
Male Pairs: 442825
Female Pairs: 547080
Mixed Pairs: 2279631
Ratio of Male to Female Pairs: 0.8094337208452146
Ratio of Female to Male Pairs: 1.2354316039067352
Ratio of Gendered to Mixed pairs: 0.4342391378253761


In [98]:
print("*** Select Female Leaning Attributes 85% Dissimilar (3 attributes different) ***")
print(f"Total Pairs: {mixed_count + male_male_count + female_female_count}")
print(f"Male Pairs: {male_male_count}")
print(f"Female Pairs: {female_female_count}")
print(f"Mixed Pairs: {mixed_count}")
print(f"Ratio of Male to Female Pairs: {male_male_count/female_female_count}")
print(f"Ratio of Female to Male Pairs: {female_female_count/male_male_count}")
print(f"Ratio of Gendered to Mixed pairs: {(male_male_count + female_female_count) / mixed_count}")

*** Select Female Leaning Attributes 85% Dissimilar (3 attributes different) ***
Total Pairs: 2748654
Male Pairs: 371391
Female Pairs: 323582
Mixed Pairs: 2053681
Ratio of Male to Female Pairs: 1.1477492567571743
Ratio of Female to Male Pairs: 0.8712704400483587
Ratio of Gendered to Mixed pairs: 0.3384035787447028


In [116]:
print("*** Select Female Leaning Attributes 95% Dissimilar (1 attribute different) ***")
print(f"Total Pairs: {mixed_count + male_male_count + female_female_count}")
print(f"Male Pairs: {male_male_count}")
print(f"Female Pairs: {female_female_count}")
print(f"Mixed Pairs: {mixed_count}")
print(f"Ratio of Male to Female Pairs: {male_male_count/female_female_count}")
print(f"Ratio of Female to Male Pairs: {female_female_count/male_male_count}")
print(f"Ratio of Gendered to Mixed pairs: {(male_male_count + female_female_count) / mixed_count}")

*** Select Female Leaning Attributes 95% Dissimilar (1 attribute different) ***
Total Pairs: 2256327
Male Pairs: 303634
Female Pairs: 206883
Mixed Pairs: 1745810
Ratio of Male to Female Pairs: 1.467660465093797
Ratio of Female to Male Pairs: 0.6813565015775572
Ratio of Gendered to Mixed pairs: 0.2924241469575727


In [86]:
print("*** Select Attributes 85% Dissimilar (3 attributes different) ***")
print(f"Total Pairs: {mixed_count + male_male_count + female_female_count}")
print(f"Male Pairs: {male_male_count}")
print(f"Female Pairs: {female_female_count}")
print(f"Mixed Pairs: {mixed_count}")
print(f"Ratio of Male to Female Pairs: {male_male_count/female_female_count}")
print(f"Ratio of Female to Male Pairs: {female_female_count/male_male_count}")
print(f"Ratio of Gendered to Mixed pairs: {(male_male_count + female_female_count) / mixed_count}")

*** Select Attributes 85% Dissimilar (3 attributes different) ***
Total Pairs: 3109868
Male Pairs: 364522
Female Pairs: 843449
Mixed Pairs: 1901897
Ratio of Male to Female Pairs: 0.43218025037672697
Ratio of Female to Male Pairs: 2.3138493698597067
Ratio of Gendered to Mixed pairs: 0.6351400733057574


In [80]:
print("*** Select Attributes 95% Dissimilar (1 attribute different) ***")
print(f"Total Pairs: {mixed_count + male_male_count + female_female_count}")
print(f"Male Pairs: {male_male_count}")
print(f"Female Pairs: {female_female_count}")
print(f"Mixed Pairs: {mixed_count}")
print(f"Ratio of Male to Female Pairs: {male_male_count/female_female_count}")
print(f"Ratio of Female to Male Pairs: {female_female_count/male_male_count}")
print(f"Ratio of Gendered to Mixed pairs: {(male_male_count + female_female_count) / mixed_count}")

*** Select Attributes 95% Dissimilar (1 attribute different) ***
Total Pairs: 2586235
Male Pairs: 307459
Female Pairs: 618695
Mixed Pairs: 1660081
Ratio of Male to Female Pairs: 0.49694760746409783
Ratio of Female to Male Pairs: 2.012284564771238
Ratio of Gendered to Mixed pairs: 0.5578968737067649


In [72]:
print("*** Select Attributes 65% Dissimilar (7 attributes different) ***")
print(f"Total Pairs: {mixed_count + male_male_count + female_female_count}")
print(f"Male Pairs: {male_male_count}")
print(f"Female Pairs: {female_female_count}")
print(f"Mixed Pairs: {mixed_count}")
print(f"Ratio of Male to Female Pairs: {male_male_count/female_female_count}")
print(f"Ratio of Female to Male Pairs: {female_female_count/male_male_count}")
print(f"Ratio of Gendered to Mixed pairs: {(male_male_count + female_female_count) / mixed_count}")

*** Select Attributes 65% Dissimilar ***
Total Pairs: 4170618
Male Pairs: 488805
Female Pairs: 1417154
Mixed Pairs: 2264659
Ratio of Male to Female Pairs: 0.344920170990591
Ratio of Female to Male Pairs: 2.8992215709741105
Ratio of Gendered to Mixed pairs: 0.8416097081282435


In [65]:
print("*** Select Attributes ***")
print(f"Total Pairs: {mixed_count + male_male_count + female_female_count}")
print(f"Male Pairs: {male_male_count}")
print(f"Female Pairs: {female_female_count}")
print(f"Mixed Pairs: {mixed_count}")
print(f"Ratio of Male to Female Pairs: {male_male_count/female_female_count}")
print(f"Ratio of Female to Male Pairs: {female_female_count/male_male_count}")
print(f"Ratio of Gendered to Mixed pairs: {(male_male_count + female_female_count) / mixed_count}")

*** Select Attributes ***
Total Pairs: 2724710
Male Pairs: 319311
Female Pairs: 659581
Mixed Pairs: 1745818
Ratio of Male to Female Pairs: 0.48411188315006043
Ratio of Female to Male Pairs: 2.065638202254229
Ratio of Gendered to Mixed pairs: 0.5607067861598403


In [92]:
print("*** All Attributes 85% Dissimilar (3 attributes different) ***")
print(f"Total Pairs: {mixed_count + male_male_count + female_female_count}")
print(f"Male Pairs: {male_male_count}")
print(f"Female Pairs: {female_female_count}")
print(f"Mixed Pairs: {mixed_count}")
print(f"Ratio of Male to Female Pairs: {male_male_count/female_female_count}")
print(f"Ratio of Female to Male Pairs: {female_female_count/male_male_count}")
print(f"Ratio of Gendered to Mixed pairs: {(male_male_count + female_female_count) / mixed_count}")

*** All Attributes 85% Dissimilar (3 attributes different) ***
Total Pairs: 2809772
Male Pairs: 329367
Female Pairs: 255679
Mixed Pairs: 2224726
Ratio of Male to Female Pairs: 1.2882051322165684
Ratio of Female to Male Pairs: 0.77627388293302
Ratio of Gendered to Mixed pairs: 0.26297440673593064
